In [3]:
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By

def generate_month_list(start_ym: str, count: int):
    """
    2026-06부터 시작해서 1개월씩 이전 달로 거슬러 올라가는 연월 리스트 생성
    """
    year, month = map(int, start_ym.split("-"))
    month_list = []
    
    for _ in range(count):
        month_list.append(f"{year}-{month:02d}")
        month -= 1
        if month == 0:
            month = 12
            year -= 1
            
    return month_list

def check_multi_crawling():
    # 1. 277부터 255까지 2씩 감소하는 ID 리스트 (총 12개)
    page_ids = list(range(277, 254, -2))
    
    # 2. 2026-06부터 2025-07까지 12개월 연월 리스트 생성
    month_list = generate_month_list("2026-06", len(page_ids))
    
    # URL ID와 연월 짝지어 매핑 (예: 277 -> '2026-06', 275 -> '2026-05', ...)
    target_tasks = list(zip(page_ids, month_list))

    # 웹드라이버 실행 (반복문 밖에서 1번만 띄워서 재사용)
    driver = webdriver.Chrome()

    try:
        print(f"🚀 총 {len(target_tasks)}개 페이지 연속 크롤링 시작...\n")

        for page_id, standard_month in target_tasks:
            target_url = f"https://www.carisyou.com/theme/top10/{page_id}"
            company_type = "수입"

            print(f"📌 [작업 진행 중] ID: {page_id} | 기준월: {standard_month} | URL: {target_url}")
            driver.get(target_url)
            time.sleep(2)  # 페이지 로딩 대기

            items = driver.find_elements(By.CSS_SELECTOR, "div.sale_rank_list ol > li")
            car_data_list = []

            for item in items:
                try:
                    # 모델명 추출
                    car_info_element = item.find_element(By.CSS_SELECTOR, "span.car_info")
                    model_name = car_info_element.text.strip()

                    # 등록 개수 추출
                    sale_count_element = item.find_element(By.CSS_SELECTOR, "span.sale_count span")
                    count_text = sale_count_element.text.strip().replace(",", "")
                    count_car_month = int(count_text) if count_text.isdigit() else 0

                    # 제조사 추출
                    company_name = model_name.split()[0] if " " in model_name else model_name

                    car_data_list.append({
                        "company_type": company_type,
                        "company_name": company_name,
                        "model_name": model_name,
                        "count_car_month": count_car_month,
                        "standard_month": standard_month
                    })

                except Exception:
                    continue

            # --- 개별 월 수집 결과 출력 ---
            print(f"   └ 수집 완료: 상위 {len(car_data_list)}개 모델")
            if car_data_list:
                # 1위, 2위 모델만 간략하게 출력 확인
                print(f"     • 1위: {car_data_list[0]['model_name']} ({car_data_list[0]['count_car_month']:,}대)")
                if len(car_data_list) > 1:
                    print(f"     • 2위: {car_data_list[1]['model_name']} ({car_data_list[1]['count_car_month']:,}대)")
            print("-" * 65)

            # 서버 부하 방지를 위한 짧은 휴식
            time.sleep(1)

        print("\n🎉 모든 기간 데이터 크롤링 확인이 완료되었습니다!")

    except Exception as e:
        print(f"❌ 크롤링 중 오류 발생: {e}")

    finally:
        driver.quit()

if __name__ == "__main__":
    check_multi_crawling()

🚀 총 12개 페이지 연속 크롤링 시작...

📌 [작업 진행 중] ID: 277 | 기준월: 2026-06 | URL: https://www.carisyou.com/theme/top10/277
   └ 수집 완료: 상위 10개 모델
     • 1위: 테슬라 모델 Y (9,188대)
     • 2위: 비야디 돌핀 (2,828대)
-----------------------------------------------------------------
📌 [작업 진행 중] ID: 275 | 기준월: 2026-05 | URL: https://www.carisyou.com/theme/top10/275
   └ 수집 완료: 상위 10개 모델
     • 1위: 테슬라 모델 Y (8,762대)
     • 2위: BMW 5시리즈 (2,060대)
-----------------------------------------------------------------
📌 [작업 진행 중] ID: 273 | 기준월: 2026-04 | URL: https://www.carisyou.com/theme/top10/273
   └ 수집 완료: 상위 10개 모델
     • 1위: 테슬라 모델 Y (10,086대)
     • 2위: 테슬라 모델 3 (2,596대)
-----------------------------------------------------------------
📌 [작업 진행 중] ID: 271 | 기준월: 2026-03 | URL: https://www.carisyou.com/theme/top10/271
   └ 수집 완료: 상위 10개 모델
     • 1위: 테슬라 모델 Y (6,751대)
     • 2위: 테슬라 모델 3 (3,702대)
-----------------------------------------------------------------
📌 [작업 진행 중] ID: 269 | 기준월: 2026-02 | URL: https://www.caris

In [ ]:
import time
import uuid
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from sqlalchemy import text

# db.py에서 SQLAlchemy 엔진 가져오기
from db import get_engine

def generate_month_list(start_ym: str, count: int):
    year, month = map(int, start_ym.split("-"))
    month_list = []
    
    for _ in range(count):
        month_list.append(f"{year}-{month:02d}")
        month -= 1
        if month == 0:
            month = 12
            year -= 1
            
    return month_list

def save_multi_crawling_to_db():
    # 1. 277부터 255까지 2씩 감소하는 ID 리스트 (총 12개)
    page_ids = list(range(276, 253, -2))
    
    # 2. 2026-06부터 2025-07까지 12개월 연월 리스트 생성
    month_list = generate_month_list("2026-06", len(page_ids))
    
    # URL ID와 연월 짝지어 매핑
    target_tasks = list(zip(page_ids, month_list))

    # DB 엔진 가져오기
    engine = get_engine()
    driver = webdriver.Chrome()

    try:
        print(f"🚀 총 {len(target_tasks)}개 페이지 데이터 수집 및 DB 저장 시작...\n")

        for page_id, standard_month in target_tasks:
            target_url = f"https://www.carisyou.com/theme/top10/{page_id}"
            company_type = "국산"

            print(f"📌 [처리 중] ID: {page_id} | 기준월: {standard_month}")
            driver.get(target_url)
            time.sleep(2)  # 페이지 로딩 대기

            items = driver.find_elements(By.CSS_SELECTOR, "div.sale_rank_list ol > li")
            car_data_list = []

            for item in items:
                try:
                    # 1. 모델명 추출
                    car_info_element = item.find_element(By.CSS_SELECTOR, "span.car_info")
                    model_name = car_info_element.text.strip()

                    # 2. 월별 등록 개수 추출 (DB 타입이 VARCHAR이므로 문자열 형태 그대로 저장)
                    sale_count_element = item.find_element(By.CSS_SELECTOR, "span.sale_count span")
                    count_car_month = sale_count_element.text.strip().replace(",", "")

                    # 3. 제조사 추출
                    company_name = model_name.split()[0] if " " in model_name else model_name

                    # 4. regist_id 고유값 생성 (UUID 활용)
                    # PK인 regist_id가 VARCHAR(255) 지정되었으므로 중복 없는 고유 문자열 생성
                    regist_id = str(uuid.uuid4())

                    car_data_list.append({
                        "regist_id": regist_id,             # PK 고유 키
                        "company_type": company_type,         # '수입'
                        "company_name": company_name,         # '테슬라', 'BMW' 등
                        "model_name": model_name,             # '테슬라 모델 Y'
                        "count_car_month": count_car_month,   # '9188' (VARCHAR)
                        "standard_month": standard_month      # '2026-06'
                    })

                except Exception:
                    continue

            if car_data_list:
                df = pd.DataFrame(car_data_list)
                with engine.begin() as conn:
                    # 재실행 시 중복 적재 방지를 위해 동일 기준월 & 구분 데이터 삭제
                    conn.execute(
                        text("""
                            DELETE FROM car_registration 
                            WHERE standard_month = :s_month AND company_type = :c_type
                        """),
                        {"s_month": standard_month, "c_type": company_type}
                    )

                df.to_sql(
                    name="car_registration",
                    con=engine,
                    if_exists="append",
                    index=False
                )

                print(f"   └  [{standard_month}] {len(df)}건 DB 저장 완료!")
            else:
                print(f"   └ ⚠️ 수집된 데이터가 없습니다.")

            print("-" * 65)
            time.sleep(1)

        print("\n🎉 모든 기간 데이터의 크롤링 및 DB 저장이 완벽하게 완료되었습니다!")

    except Exception as e:
        print(f"❌ 작업 중 오류 발생: {e}")

    finally:
        driver.quit()

if __name__ == "__main__":
    save_multi_crawling_to_db()

🚀 총 12개 페이지 데이터 수집 및 DB 저장 시작...

📌 [처리 중] ID: 276 | 기준월: 2026-06
   └  [2026-06] 10건 DB 저장 완료!
-----------------------------------------------------------------
📌 [처리 중] ID: 274 | 기준월: 2026-05
   └  [2026-05] 10건 DB 저장 완료!
-----------------------------------------------------------------
📌 [처리 중] ID: 272 | 기준월: 2026-04
   └  [2026-04] 10건 DB 저장 완료!
-----------------------------------------------------------------
📌 [처리 중] ID: 270 | 기준월: 2026-03
   └  [2026-03] 10건 DB 저장 완료!
-----------------------------------------------------------------
📌 [처리 중] ID: 268 | 기준월: 2026-02
   └  [2026-02] 10건 DB 저장 완료!
-----------------------------------------------------------------
📌 [처리 중] ID: 266 | 기준월: 2026-01
   └  [2026-01] 10건 DB 저장 완료!
-----------------------------------------------------------------
📌 [처리 중] ID: 264 | 기준월: 2025-12
   └  [2025-12] 10건 DB 저장 완료!
-----------------------------------------------------------------
📌 [처리 중] ID: 262 | 기준월: 2025-11
   └  [2025-11] 10건 DB 저장 완료!
--------

In [ ]:
import os
import re
import time
import pymysql
from selenium import webdriver
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--start-maximized")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

DB_USER = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_DATABASE", "cars_db")

url = "https://www.encar.com/mocha.do"


def get_db_connection():
    return pymysql.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME,
        port=int(DB_PORT),
        charset="utf8mb4",
    )


def clean_car_name(raw_name):
    # 1. 괄호와 그 안에 든 내용 제거 (예: "(18년~18년)" 제거)
    name = re.sub(r"\([^)]*\)", "", raw_name)
    # 2. 불필요한 세부 스펙 키워드 및 연식/배기량 단위 제거
    name = re.sub(r"(가솔린|디젤|하이브리드|전기|LPI|\d+\.\d+[L]|[\d~년]+)", "", name)
    # 3. 하이픈 제거 (예: "S-클래스" -> "S클래스")
    name = name.replace("-", "")
    # 4. 공백 정리
    name = re.sub(r"\s+", " ", name).strip()
    return name


def click_until_hidden(driver, timeout=10):
    button_locator = (By.CSS_SELECTOR, "#btnInfoMore")

    while True:
        try:
            button = driver.find_element(*button_locator)

            is_displayed = driver.execute_script(
                "return window.getComputedStyle(arguments[0]).display !== 'none';",
                button,
            )

            if not is_displayed:
                print("버튼이 display: none 상태가 되어 클릭을 중단합니다.")
                break

            try:
                button.click()
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", button)

            print("더보기 버튼 클릭 완료")
            time.sleep(0.5)

        except (NoSuchElementException, StaleElementReferenceException):
            print("더보기 버튼을 찾을 수 없거나 끝까지 로딩되었습니다.")
            break


def for_in_print(elem_lists, title):
    for elem in elem_lists:
        text = elem.text.strip()
        if text:
            print(f"[{title}:] {text}")


def insert_review_datas(
    index, cursor, title, content, score, domain_type
):
    sql_total_review = """
                        INSERT INTO total_review
                        (total_review_id, review_id2, total_review_title, total_review_content, total_score, domain_type)
                        VALUES (%s, %s, %s, %s, %s, %s)
                        """
    total_review_id = f"{index}_{domain_type}"
    total_review_data = (
        total_review_id,
        str(index),
        title,
        content,
        score,
        domain_type,
    )
    cursor.execute(sql_total_review, total_review_data)


def print_detail_texts(driver, index):
    connection = get_db_connection()
    try:
        car_name_selector = "#mocha_car > h3 > a"
        area_detail = "#depth_main > div > div.box_g.box_total > div.area_detail"
        area_review = "#depth_main > div > div.box_g.box_total > div.area_review"
        perform_detail = (
            "#depth_main > div > div.box_g.box_perform > div.area_summary"
        )
        price_detail = "#depth_main > div > div.box_g.box_price > div.area_summary"
        fault_detail = "#depth_main > div > div.box_g.box_faulty > div.area_summary"

        car_name_elem = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, car_name_selector))
        )
        area_detail_container = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, area_detail))
        )
        area_review_container = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, area_review))
        )
        perform_detail_container = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, perform_detail))
        )
        price_detail_container = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, price_detail))
        )
        fault_detail_container = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, fault_detail))
        )

        area_detail_elements = area_detail_container.find_elements(
            By.CSS_SELECTOR, "strong, p"
        )
        area_review_elements = area_review_container.find_elements(
            By.CSS_SELECTOR, "strong"
        )
        perform_detail_elements = perform_detail_container.find_elements(
            By.CSS_SELECTOR, "strong, span"
        )
        price_detail_elements = price_detail_container.find_elements(
            By.CSS_SELECTOR, "strong, span"
        )
        fault_detail_elements = fault_detail_container.find_elements(
            By.CSS_SELECTOR, "strong, span"
        )

        # 데이터 파싱 및 정제
        raw_car_name = car_name_elem.text.strip()
        car_name = clean_car_name(raw_car_name)

        detail_title = (
            area_detail_elements[0].text.strip()
            if len(area_detail_elements) > 0
            else ""
        )
        detail_content = (
            area_detail_elements[1].text.strip()
            if len(area_detail_elements) > 1
            else ""
        )
        detail_score = (
            area_review_elements[0].text.strip()
            if len(area_review_elements) > 0
            else "0"
        )

        perform_title = (
            perform_detail_elements[0].text.strip()
            if len(perform_detail_elements) > 0
            else ""
        )
        perform_score = (
            perform_detail_elements[1].text.strip()
            if len(perform_detail_elements) > 1
            else "0"
        )

        price_title = (
            price_detail_elements[0].text.strip()
            if len(price_detail_elements) > 0
            else ""
        )
        price_score = (
            price_detail_elements[1].text.strip()
            if len(price_detail_elements) > 1
            else "0"
        )

        fault_title = (
            fault_detail_elements[0].text.strip()
            if len(fault_detail_elements) > 0
            else ""
        )
        fault_score = (
            fault_detail_elements[1].text.strip()
            if len(fault_detail_elements) > 1
            else "0"
        )

        try:
            with connection.cursor() as cursor:
                # CAR_REGISTRATION 테이블에서 정제된 이름으로 유연하게 검색
                find_id_sql = """
                    SELECT regist_id 
                    FROM CAR_REGISTRATION 
                    WHERE model_name LIKE %s 
                    LIMIT 1
                """
                search_query = f"%{car_name}%"
                cursor.execute(find_id_sql, (search_query,))
                matched_car = cursor.fetchone()

                if matched_car:
                    real_regist_id = matched_car[0]
                else:
                    real_regist_id = "01dd5a88-1287-4fce-b573-9108b8b6ef23"
                    print(f"경고: '{car_name}' (원본: {raw_car_name}) 매칭 실패로 기본 UUID 대체")

                real_model_id = 1  # 모델 고정값

                # 1. review 테이블 INSERT
                sql_review = """
                           INSERT INTO review (review_id, model_id, regist_id, brand_name_review)
                           VALUES (%s, %s, %s, %s)
                           ON DUPLICATE KEY UPDATE brand_name_review = VALUES(brand_name_review)
                           """
                review_data = (str(index), real_model_id, real_regist_id, car_name)
                cursor.execute(sql_review, review_data)

                # 2. total_review 테이블 INSERT
                insert_review_datas(
                    index, cursor, detail_title, detail_content, detail_score, "1"
                )
                insert_review_datas(
                    index, cursor, perform_title, "", perform_score, "2"
                )
                insert_review_datas(index, cursor, price_title, "", price_score, "3")
                insert_review_datas(index, cursor, fault_title, "", fault_score, "4")

                connection.commit()
                print(f"[{index}번째 차량: {car_name}] 데이터 삽입 완료! (regist_id: {real_regist_id})")

        except Exception as e:
            connection.rollback()
            print(f"데이터 삽입 실패: {e}")

    except TimeoutException:
        print("페이지 요소 로딩 시간 초과")
    finally:
        connection.close()


def open_links_and_scrape(driver, start_idx):
    main_window = driver.current_window_handle
    links = driver.find_elements(By.CSS_SELECTOR, "#list_mocha li a")
    href_list = [
        link.get_attribute("href") for link in links if link.get_attribute("href")
    ]

    print(f"총 {len(href_list)}개의 링크를 발견했습니다.")
    current_idx = start_idx

    for href in href_list:
        print(f"\n--- [전체 순번: {current_idx}] 링크 처리 중 ---")
        driver.execute_script("window.open(arguments[0], '_blank');", href)
        all_windows = driver.window_handles
        driver.switch_to.window(all_windows[-1])

        print_detail_texts(driver, current_idx)

        driver.close()
        driver.switch_to.window(main_window)
        current_idx += 1
        time.sleep(0.3)

    return current_idx


# 메인 실행 루프
try:
    global_idx = 1

    for i in ["001","002","003","087","013","012","035","017","031","011","054"]:
        target_url = f"{url}?mnfccd={i}"
        print(f"\n==========================================")
        print(f"페이지 이동: {target_url}")
        print(f"==========================================")

        driver.get(target_url)
        time.sleep(1)

        click_until_hidden(driver)
        global_idx = open_links_and_scrape(driver, global_idx)

finally:
    driver.quit()


페이지 이동: https://www.encar.com/mocha.do?mnfccd=001
더보기 버튼 클릭 완료
더보기 버튼 클릭 완료
버튼이 display: none 상태가 되어 클릭을 중단합니다.
총 74개의 링크를 발견했습니다.

--- [전체 순번: 1] 링크 처리 중 ---
[strong] "마음을 홀리는 자동차"
[p] 뽀빠이 과자 먹다가 별사탕 하나가 입안에 턱 들어왔을 때의 느낌, 기억하시나요? 정신이 번쩍 뜨이는 단맛과 함께 침이 솟구치며 텁텁했던 입안이 부드럽게 풀리곤 했죠. 저는 캐스퍼가 그런 존재가 아닌가 합니다. 새까만 아스팔트 위 잿빛 SUV와 세단만 가득한 우리네 도로 위에, 앙증맞은 캐스퍼가 나타나면 그 퍽퍽함이 조금은 해소되는 느낌이거든요.

작고 귀여워서 그렇습니다. 소형 SUV는 옛 준중형 SUV보다 더 커지고 아반떼는 옛 쏘나타만큼 거대한 요즘에, 경차 규격에 딱 걸린 캐스퍼는 유달리 조그맣죠. 더구나 기아 모닝이나 레이마저 사이버틱하게 변해버려서 캐스퍼는 홀로 레트로 분위기가 풍길 정도로 귀엽습니다. 동글동글한 눈망울 같은 헤드램프가 대표적이죠.

그런데 이토록 귀여운 차가 의외로 내공도 좋다는 사실, 알고 계신가요? 실내에 앉아 보면 꼴에 SUV라고 개방감이 좋고, 차급이 의심스러울 정도의 편의장비를 품고 있으며, 과감히 4인승으로 꾸린 실내는 성인 남성 넷을 편안히 태울 수 있습니다. 시트를 잘 접으면 길이가 2m 넘는 짐까지 꿀꺽 삼키죠. 작은 공간을 다재다능하게 활용하는 공간 활용성을 보면 이보다 큰 차가 낭비로 느껴질 정도입니다.

물론 2000년대 초반에 머물러있는 듯한 파워트레인 등 단점이 없지는 않지만, 귀여운 외모부터 톡톡 튀는 공간 활용성까지 여러모로 캐스퍼는 마음을 홀리는 매력을 품은 자동차입니다. 그 매력을 이차어때가 자세히 파헤쳐 봤습니다.
[종합 리뷰:] 3.5
[성능:] "터보 토크가 좋다"
[성능:] 2.0
[가격:] "새 시대를 열다"
[가격:] 3.5
[문제점:] "통상적인 수준"
[문제점